# Spin Pumping of an exchange coupled spin-1/2 pair.
This notebook simulates the transfer of angular momentum which is observed in STM experiments. It contains:
- Spin Hamiltonian
- Voltage dependent Magnetization

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import SpinSys, ESR, bath, hamil
from spinfinity import SpinPumping as sp


%load_ext autoreload
%autoreload 2

## Spin Hamiltonian
The two spin-1/2 experience an external field of 0.5 T along z.
The tip field acts only on the readout spin.

The exchange interaction is parameterized relative to the z axis:

$$
H_{\mathrm{ex}}
=
J_\perp(S_{0x}S_{1x}+S_{0y}S_{1y})
+
J_\parallel S_{0z}S_{1z}.
$$

In [ ]:
# Define the spin system
spin = SpinSys([0.5, 0.5])

spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.05]  # T
spin.gfactorvector = [2.0, 2.0]
spin.ReadoutSpin = 0
spin.tip = "f"


# Exchange parameters expressed as energy / h in GHz
J_perp_GHz = 0.4
J_parallel_GHz = 0.2

# Convert to the meV units required by the Hamiltonian
exchange_meV = np.array([
    J_perp_GHz,
    J_perp_GHz,
    J_parallel_GHz,
]) * spin.GHztomeVConversion

hamil.assign_ExchangeInteraction(
    spin,
    [(0, 1, exchange_meV)],
)

# Do not add a separate dipolar interaction
spin.DipoleVector[:] = 0.0

hamil.calc_EigEnergies(spin, rebuildH=True)
hamil.calc_EigStates(spin)
hamil.plot_EigenMatrix(spin);


## cw-ESR
The exchagne coupling parameters are visible in a Tipfield dependent ESR map.

In [ ]:
freq_array=np.linspace(11,17,1000)
lw = 0.02
ESR.plot_FreqSweep(spin,
        freq_array=freq_array,
        AllowPumping=False,
        lw=lw,
        norm=True,
        output=False);

Btip_array = np.linspace(-0.04,0.04,500)
ESR.plot_Tipfieldmap(
        spin,
        freq_array=freq_array,
        BTip_array=Btip_array,
        lw=lw,
        norm=True,
        AllowPumping=False,
        Angle=0.0);

## Voltage dependent magnetization

In [ ]:
# Tunneling Parameters
spin.T = 0.2  # K
eta = 0.4
spin.TipPolarization = [0.0, 0.0, eta]

spin.G_st = 0.5e-9  # S
conductance_ratio = 0.2
G_ss = spin.G_st / conductance_ratio
spin.G_ss = [G_ss, G_ss]
spin.G_tt = 1  # Enable tip-tip scattering; this is a dimensionless switch
spin.U[:] = 0.0
spin.b0 = 0.0

voltage_array = np.linspace(-100, 100, 2001)

# Plotting the magnetization of the remote spin
sp.plot_Magnetization_Voltage(spin, V_array=voltage_array, spins=[1]);

## Looking at the populations
if we consider the populations of the first spin A and the second spin seperately we can use the sub system of a spin 1/2 to compute the populations.

In [ ]:
# subspin system
spin1 = SpinSys([0.5])
spin1.B = [0.0,0.0,0.5]
hamil.calc_EigEnergies(spin1)

sp.plot_SplitPopulations_Voltage(spin, spin1, voltage_array);